### RAG (Retrieval Augemented Generation)
- 기존의 LLM을 확장하여 새로운 콘텐츠나 질문에 더욱 정확하고 확실한 정보를 제공하는 방법
- 모델이 학습하지 않은 외부데이터에 대해 환각(Hallucination) 발생을 방지하기 위함
- 외부데이터를 실시간으로 검색하여 활용하여 답변을 생성하는 기술

### 기본구조
- 검색단계 : 사용자의 질문이나 컨텍스트를 입력받아 관련된 외부데이터를 검색하는 단계
- 증강단계 : 검색한 데이터를 토큰화, 인코딩, 임베딩 후 벡터DB에 저장하여 검색기를 붙이는 단계
- 생성단계 : 벡터 DB에 저장된 데이터와 LLM을 사용하여 사용자의 질문에 답하는 단계

### 장점
- 풍부한 정보 제공 : 검색한 겨로가를 활용하여 답변하기 때문에 구체적이고 풍부한 정보를 제공
- 실시간 정보 반영 : 최신데이터를 검색하여 답변하므로 모델이 실시간으로 변화하는 정보에 대응이 가능
- 환각 방지 : 외부데이터의 검색을 통한 답변을 생성함으로써, 환각현상의 발생을 줄임

#### 프로세스 
- 사전준비단계
    - 문서불러오기(pdf)
    - 텍스트분할
    - 임베딩
    - 벡터DB저장
- 실행단계
    - 검색기 설정
    - 프롬프트 구성
    - LLM 생성
    - 체인 생성 및 실행


### RAG1 : PDF 파일을 학습한 나만의 챗봇 만들기
1. 문서불러오기 (데이터 로드)
    - RAG에 사용할 데이터를 불러오는 단계
2. 텍스트 분할 (Text Split)
    - 불러온 데이터를 작은 크기 단위인 청크 (Chunk)로 분할
3. 임베딩 (Embedding)
    - 분할된 텍스트데이터들을 검색 가능한 형태로 만드는 관계
4. 검색 
    - 사용자의 질문에 대답하기 위해 가장 관련있는 정보를 찾는 단계 (유사도 검색)
5. 생성
    - 검색한 정보를 기반으로 사용자의 질문에 답변을 생성

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# 작업 디렉토리 설정 (본인 경로로 변경)
%cd /content/drive/Othercomputers/LG DX 노트북/LG_Dx_School/Deep Learning
!pwd

/content/drive/Othercomputers/LG DX 노트북/LG_Dx_School/Deep Learning
/content/drive/Othercomputers/LG DX 노트북/LG_Dx_School/Deep Learning


In [3]:
# 앞으로 사용할때에는
# 구글 마운트 -> 저장된 api_key 불러서 사용
import os
with open('./key/.openai_api_key','r') as f:
  api_key = f.read().strip()

# 환경변수 설정 (딕셔너리형태)
os.environ['OPENAI_API_KEY'] = api_key

In [4]:
!pip install -qU langchain langchain_openai langchain_core langchain_community
!pip install -qU tiktoken pypdf chromadb faiss-cpu
!pip install -qU langchain-teddynote

# langchain-openai    : OpenAI의 LLM을 LangChain과 통합하기 위한 확장 패키지
# langchain           : LangChain의 핵심 라이브러리, LLM 기반 애플리케이션 개발을 위한 프레임워크 제공
# langchain_community : 커뮤니티에서 제공하는 LangChain 확장 및 추가 도구 모음
# tiktoken            : OpenAI에서 제공하는 토크나이저 라이브러리로, 토큰 수 계산 등에 사용됨
# pypdf               : PDF 파일에서 텍스트를 추출하기 위한 라이브러리
# chromadb            : 벡터 임베딩 저장 및 검색을 위한 벡터 데이터베이스 (Chroma DB)
# faiss-cpu           : 효율적인 벡터 유사도 검색을 위한 FAISS 라이브러리의 CPU 버전
# langchain-teddynote : 테디노트(유튜버)가 만든 랭체인 확장 패키지

In [5]:
from langchain_community.document_loaders import PyPDFLoader

In [7]:
loader = PyPDFLoader('./data/미래 필수 역량.pdf')

In [9]:
document = loader.load()

2. 텍스트 분할 (Text split)
    - 작은크기의 단위로 분할
    - chunk : 하나의 문서를 일정한 길이로 잘라는 조각

#### 분할도구 : RecursiveChracterTextSplit
- 문단 -> 문장 -> 단어 -> 문자 순으로 재귀적 분할
- 텍스트의 구조를 고려하여 분할 
- 문맥이 잘리는 단점을 보완

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_spliter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50,
    length_function = len
)

In [14]:
# 적용, 청크화
chunk = text_spliter.split_documents(document)

In [15]:
for i,text in enumerate(chunk):
    print(f"결과 {i+1}\n {text}")
    print()

결과 1
 page_content='홈 • 인공지능 • 일문일답 | AI 혁명 속 승자가 되는 법··· AW S 교육 임원이 말하는 미래 필수 역량
By Lucas M earian
Senior Reporter
일문일답 | AI 혁명 속 승자가 되는 법···AWS 교육 임원이 말하는 미래 필수 역량
인터뷰
2025.03.03 • 11분
교육 산업 생성형 AI IT 직업
AWS 교육 및 수료증 프로그램에 대한 수요가 급증하고 있다. 일부 과정은 수강생이 전년 대비 9
배까지 증가했다. 아마존웹서비스(AWS)의 교육·인증 제품 및 서비스 디렉터 제니 트라우트먼은
이러한 수요 증가는 최근 급변하는 기술 역량에 대한 시장의 요구를 반영한다고 설명한다.
CREDIT: JENNY TROUTMAN / JENNY TROUTMAN' S LINKEIN' metadata={'producer': 'Skia/PDF m133', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/133.0.0.0 Safari/537.36', 'creationdate': '2025-03-05T04:57:45+00:00', 'title': '일문일답 | AI 혁명 속 승자가 되는 법··· AWS 교육 임원이 말하는 미래 필수 역량 | CIO', 'moddate': '2025-03-05T04:57:45+00:00', 'source': './data/미래 필수 역량.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}

결과 2
 page_content='AI와 기술 역량이 빠르게 변화하면서 취업이나 커리어 발전을 위해 필요한 역량도 몇 년 전과 비교해 크게
달라졌다. 특히 직무 능력 외에 의사소통, 문제 해결, 협업, 리더십 능력 등으로 대표되는 ‘소프트 스킬’의
중요성이 그 어느 때보다 커지고 있다고 AW S의 교육·수료(training 

In [16]:
print(len(chunk))

21


3. 임베딩 (Embedding)
- 분리된 청크를 인코딩하고 청크간의 연관성을 포함하여 벡터화하는 과정
- 임베딩 결과를 벡터 DB에 저장 
    - Faiss
    - Chroma : Python언어에 최적화 된 DB -> llm,LangChain 프레임워크와 호환성이 좋음
- 벡터 DB 
    - 데이터를 벡터형태로 저장, 유사도 기반 검색이 가능
    - 저장 + 검색 동시에 제공

In [ ]:
# 텍스트를 임베딩할 때는 모델의 임베더와 동일한 모델로 진행해야함
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma # 벡터 DB

In [19]:
# 임베더 객체 생성
embedding = OpenAIEmbeddings()

vector_store = Chroma.from_documents(documents=chunk,
                                     embedding = embedding)

4. 검색 
- 사용자의 입력을 바탕으로 쿼리를 생성하여 연관성이 높은 정보를 검색

In [20]:
# 검색기 
retriever = vector_store.as_retriever(search_kwargs = {'k':2})
# 가장 유사한 데이터 2개를 참고하여 출력

5. 생성
- llm에게 사용자의 입력과 검색한 결과를 함께 묶어 전달 
- 모델은 사전학습된 지식과 검색한 결과를 포함하여 정답을 출력

In [21]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [27]:
## 프롬프트 템플릿 생성
prompt = ChatPromptTemplate.from_messages([
    ('system','{context} 해당 문서에 있는 내용으로만 답변해줘'),
    ('human','{question}')
])

In [28]:
## 모델 생성
llm = ChatOpenAI(model = 'gpt-4o-mini',
           temperature = 0)

In [29]:
## 체인연결
rag_chain = ({'question' : RunnablePassthrough(),'context' : retriever}
             | prompt
             | llm
             | StrOutputParser())

In [ ]:
## 체인 구동 확인
rag_chain.invoke('위 파일은 어떤 내용인가요?')

'위 파일은 AI 혁명과 관련된 필수 역량에 대한 내용을 다루고 있습니다. AWS 교육 임원이 인터뷰 형식으로 AI의 활용 방법과 개인 및 업무에서 AI를 어떻게 적용할 수 있는지에 대한 의견을 제시하고 있습니다. 특히, AI를 직접 사용해보는 경험이 중요하다는 점과, 이를 통해 일상생활에서도 AI를 활용할 수 있는 방법을 이해할 수 있다는 내용을 포함하고 있습니다. 또한, AWS가 제공하는 교육 프로그램과 직원들이 AI 관련 인증을 취득하도록 지원하는 내용도 언급되고 있습니다.'

In [ ]:
## 체인 구동 > 무한반복문 활용
while True : 
    

In [30]:
rag_chain.invoke("삼성전자")

'삼성전자에 대한 구체적인 정보는 제공된 문서에 포함되어 있지 않습니다. 문서의 내용은 AI 혁명과 관련된 필수 역량, 비즈니스와 기술의 융합, 그리고 AWS의 교육 프로그램에 대한 내용입니다. 삼성전자에 대한 질문이 있다면, 다른 자료나 정보를 참고해 주시기 바랍니다.'